## Feature Engineering

##### Importing Packages

In [18]:
import pandas as pd
import numpy as np

##### Loading Data

In [19]:
df = pd.read_csv('../data/interim/customer_churn_cleaned.csv')
df.head()

,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.50,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,Yes,1,89,5340,Competitor had better devices


In [20]:
TARGET = "Churn Value"

drop_cols = [
    "CustomerID",
    "Count",

    "Churn Label",
    "Churn Score",
    "Churn Reason"
]

df_model = df.drop(
    columns = drop_cols
)

In [21]:
def create_tenure_group(tenure):
    if tenure <= 12:
        return "New"
    elif tenure <= 36:
        return "Medium"
    else:
        return "Long"
    
df_model['Tenure Group'] = (
    df_model['Tenure Months']
    .apply(create_tenure_group)
)

In [22]:
df_model.head()

,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,...,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value,CLTV,Tenure Group
0,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,...,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,3239,New
1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,...,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,2701,New
2,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,...,Yes,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1,5372,New
3,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,...,Yes,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1,5003,Medium
4,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,...,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1,5340,Long


In [23]:
service_cols = [
    "Phone Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies"
]

In [24]:
df_model['Total Services'] = (
    df_model[service_cols]
    .apply(
        lambda row:
        sum(row == "Yes"),
        axis = 1
    )
)

In [25]:
df_model.head()

,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,...,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value,CLTV,Tenure Group,Total Services
0,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,...,No,Month-to-month,Yes,Mailed check,53.85,108.15,1,3239,New,3
1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,...,No,Month-to-month,Yes,Electronic check,70.70,151.65,1,2701,New,1
2,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,...,Yes,Month-to-month,Yes,Electronic check,99.65,820.50,1,5372,New,4
3,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,...,Yes,Month-to-month,Yes,Electronic check,104.80,3046.05,1,5003,Medium,5
4,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,...,Yes,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.30,1,5340,Long,5


In [26]:
### +1 avoids division by 0 
df_model['Avg Monthly Spend'] = (
    df_model['Total Charges'] / (df_model['Tenure Months'] + 1)
)

In [27]:
median_cltv = (
    df_model['CLTV']
    .median()
)

df_model['High Value Customer'] = (
    df_model['CLTV'] > median_cltv
).astype(int)

In [28]:
df_model['Monthly Contract'] = (
    df_model['Contract'] == "Month-to-month"
).astype(int)

In [29]:
location_cols = [
    "Zip Code",
    "Lat Long",
    "Latitude",
    "Longitude"
    "Country",
    "State"
]

df_model = df_model.drop(
    columns = location_cols
)

KeyError: "['LongitudeCountry'] not found in axis"

In [30]:
df_model.head()

,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,Senior Citizen,Partner,...,Payment Method,Monthly Charges,Total Charges,Churn Value,CLTV,Tenure Group,Total Services,Avg Monthly Spend,High Value Customer,Monthly Contract
0,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,No,No,...,Mailed check,53.85,108.15,1,3239,New,3,36.050000,0,1
1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,No,No,...,Electronic check,70.70,151.65,1,2701,New,1,50.550000,0,1
2,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,No,No,...,Electronic check,99.65,820.50,1,5372,New,4,91.166667,1,1
3,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,No,Yes,...,Electronic check,104.80,3046.05,1,5003,Medium,5,105.036207,1,1
4,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,No,No,...,Bank transfer (automatic),103.70,5036.30,1,5340,Long,5,100.726000,1,1


In [31]:
df_model.columns

Index(['Country', 'State', 'City', 'Zip Code', 'Lat Long', 'Latitude',
       'Longitude', 'Gender', 'Senior Citizen', 'Partner', 'Dependents',
       'Tenure Months', 'Phone Service', 'Multiple Lines', 'Internet Service',
       'Online Security', 'Online Backup', 'Device Protection', 'Tech Support',
       'Streaming TV', 'Streaming Movies', 'Contract', 'Paperless Billing',
       'Payment Method', 'Monthly Charges', 'Total Charges', 'Churn Value',
       'CLTV', 'Tenure Group', 'Total Services', 'Avg Monthly Spend',
       'High Value Customer', 'Monthly Contract'],
      dtype='object')

In [32]:
df_model.to_csv(
    "../data/processed/customer_churn_features.csv",
    index=False
)

## Feature Engineering Summary

Created:

1. Tenure Group
   Reason:
   New customers showed higher churn tendency.

2. Total Services
   Reason:
   Measures customer engagement.

3. Avg Monthly Spend
   Reason:
   Normalizes spending by customer lifetime.

4. High Value Customer
   Reason:
   Captures business value.

5. Monthly Contract
   Reason:
   Month-to-month customers had higher churn.

Removed:
- Geographic coordinates for baseline model.


Experiments Planned:
A: Without CLTV
B: With CLTV
C: Remove correlated Total Charges